# IAF-Proximity Supplementary Analysis (Protocol-1, n=42)

**Objective.** Test whether individual alpha frequency (IAF) proximity to 10Hz is associated with rTMS responder status, in the n=42 protocol-1 subgroup - replicating Corlier et al. (2019)/Roelofs et al. (2021)'s association, per Decision 5's supplementary test structure.

**Inputs.**
- `data/features/iaf_protocol1.parquet` (42 subjects, IAF and IAF-prox, already extracted and validated)
- Age, responder status, protocol from `cohort_filtered_n163.xlsx`

**Method.** Primary: logistic regression, `responder ~ IAF_prox + age` (joint inclusion, not fold-scoped deconfounding - this is a one-shot inferential test, not a predictive/CV evaluation, so `AgeDeconfounder`'s leakage-prevention machinery doesn't apply here). Companion: Mann-Whitney U on IAF-prox alone, closer to Roelofs et al.'s original continuous-outcome design.

**Assumptions.**
- 17 non-responders / 25 responders is close to but under the conventional 10-events-per-predictor floor for a 2-predictor logistic regression (already flagged in Decision 5) - stated as a precision caveat on any result here, not a reason not to run it.
- Decision 7's age-responder association (t=-2.68, p=.01) was established on the full n=160 cohort; whether it holds in this n=42 subgroup specifically is checked directly below, before deciding how much weight age-adjustment should carry in this particular test.

In [1]:
#Imports and set up
import numpy as np
import pandas as pd
from scipy import stats
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing import find_repo_root

import statsmodels.api as sm

In [2]:
project_root = find_repo_root()
data_dir = project_root / "data"

iaf_df = pd.read_parquet(data_dir / "features" / "iaf_protocol1.parquet")
cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")

iaf_model_df = iaf_df.merge(
    cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='left'
).drop(columns='TDBRAIN_ID')

print(len(iaf_model_df))                                    # expect 42
print(iaf_model_df[['iaf', 'iaf_prox', 'age', 'Responder']].isna().sum())  # expect all 0
print(iaf_model_df['Responder'].value_counts())              # expect 17/25

# Age-responder association within this n=42 subgroup specifically -
# checking whether Decision 7's whole-cohort finding (t=-2.68, p=.01, n=160)
# holds in this particular subgroup, not assuming it transfers.
responders = iaf_model_df[iaf_model_df['Responder'] == 1]['age']
non_responders = iaf_model_df[iaf_model_df['Responder'] == 0]['age']

t_stat, p_val = stats.ttest_ind(responders, non_responders, equal_var=False)  # Welch's, matching Decision 7's test
print(f"\nAge: responders M={responders.mean():.2f}, non-responders M={non_responders.mean():.2f}")
print(f"Welch's t={t_stat:.2f}, p={p_val:.3f}")

42
iaf          0
iaf_prox     0
age          0
Responder    0
dtype: int64
Responder
1    25
0    17
Name: count, dtype: int64

Age: responders M=36.75, non-responders M=44.54
Welch's t=-2.00, p=0.055


In [3]:
# Primary: logistic regression, responder ~ IAF_prox + age (joint inclusion)
X = iaf_model_df[['iaf_prox', 'age']]
X = sm.add_constant(X)
y = iaf_model_df['Responder']

logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())

# Companion: Mann-Whitney U on IAF-prox alone, no age adjustment -
# closer to Roelofs et al.'s original continuous-outcome design
responders_iaf = iaf_model_df[iaf_model_df['Responder'] == 1]['iaf_prox']
non_responders_iaf = iaf_model_df[iaf_model_df['Responder'] == 0]['iaf_prox']

u_stat, u_p = stats.mannwhitneyu(responders_iaf, non_responders_iaf, alternative='two-sided')
print(f"\nMann-Whitney U: U={u_stat:.1f}, p={u_p:.4f}")
print(f"IAF-prox: responders M={responders_iaf.mean():.3f}, non-responders M={non_responders_iaf.mean():.3f}")

Optimization terminated successfully.
         Current function value: 0.622096
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              Responder   No. Observations:                   42
Model:                          Logit   Df Residuals:                       39
Method:                           MLE   Df Model:                            2
Date:                Tue, 15 Sep 2026   Pseudo R-squ.:                 0.07823
Time:                        12:26:10   Log-Likelihood:                -26.128
converged:                       True   LL-Null:                       -28.346
Covariance Type:            nonrobust   LLR p-value:                    0.1089
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.5519      1.317      1.937      0.053      -0.030       5.133
iaf_prox       0.1369      0.

## IAF-Proximity Supplementary - Findings

No association between IAF-proximity and responder status in this n=42 protocol-1 subgroup, by either test:

| Test | Statistic | p-value |
|---|---|---|
| Logistic regression (IAF-prox coefficient, age-adjusted) | coef=0.137, z=0.357 | 0.721 |
| Mann-Whitney U (IAF-prox alone) | U=236.5 | 0.546 |

Direction is opposite to Corlier et al. (2019)/Roelofs et al. (2021)'s finding: responders show slightly *higher* mean IAF-prox (1.218, further from 10Hz) than non-responders (1.115), rather than lower. Given p=0.72, this is noise around a null effect, not a reversed finding - stated for completeness, since replication fidelity is the point of this test.

Age's coefficient (-0.057, p=0.052) is consistent with the direct subgroup-level check run earlier in this notebook (Welch's t=-2.00, p=0.055): borderline but not quite significant at n=42, same direction as the full-cohort finding (Decision 7), and with a comparable or slightly larger effect size - read as an underpowered replication of the same confound, not evidence it's weaker in this subgroup.

This is a different kind of result from the Bailey supplementary null. IAF-proximity, unlike Bailey's construct, is one of only two biomarkers Klooster et al. (2024) certify as robust in their review - positively replicated by both Corlier (2019) and Roelofs (2021) before this test. A null result here is a genuine non-replication of a previously well-supported finding, in a smaller sample (n=42) than either prior study, not a third instance of an already-failing construct.

**Residual assumptions**

The 17/25 responder split with 2 predictors sits under the conventional 10-events-per-predictor floor for logistic regression (already flagged in Decision 5); standard errors in the fitted model are reasonable and no convergence/separation issues occurred, but precision at this sample size remains limited.

No claim is made about *why* the direction reversed relative to Corlier/Roelofs - p=0.72 means this is consistent with no true effect, and no further explanation is warranted or attempted here.

Roelofs et al.'s (2021) own replication sample has unconfirmed but plausible overlap with this cohort (Decision 5) - this test's independence from that prior replication is not fully established, unlike its independence from Corlier (2019).
